# 04 — ICL tracks Bayes-optimal on a synthetic HMM mixture

**Programme 04 — ICL as Implicit Bayes / Meta-Learning.** [Programme file.](../programmes/04-icl-as-bayes-meta-learning.md)

By the end of this notebook you will have built a small mixture of HMMs, generated a synthetic pretraining dataset, trained a tiny transformer on next-token prediction, and compared the trained model's next-token predictions to the *exact* Bayes-optimal predictor under the data-generating mixture. The claim being demonstrated is that, on synthetic mixture-of-HMM data, a trained transformer's in-context predictions track the Bayes-optimal posterior as context length grows — the central result of Xie, Raghunathan, Liang, Ma (2022, [arXiv:2111.02080](https://arxiv.org/abs/2111.02080)) on a budget-toy scale.

What this notebook is *not*: evidence that ICL on natural language is Bayes-optimal. The cleanest contestation of the Bayesian story (Panwar et al. 2023; Singh et al. 2023) shows it as a useful first-order model with non-trivial second-order deviations on real data.

In [ ]:
# Install pinned versions (Colab).
# %pip install -q torch==2.4.1 numpy==1.26.4 matplotlib==3.9.2

In [ ]:
import os, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 1
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
os.makedirs('figures/04', exist_ok=True)

## Define a mixture of 4 HMMs over a 4-symbol alphabet

Each task is an HMM with 3 hidden states; we have 4 tasks. The mixture prior is uniform. The pretraining distribution is: pick a task uniformly, then sample a length-$T$ sequence from that task's HMM. The Bayes-optimal next-symbol predictor at each position averages over the posterior $p(\text{task} \mid \text{prefix})$.

In [ ]:
VOCAB = 4
STATES = 3
N_TASKS = 4
T = 64

def make_hmm(rng):
    # transition matrix (S x S), each row a probability simplex.
    A = rng.dirichlet(alpha=np.full(STATES, 0.5), size=STATES)
    # emission matrix (S x V), each row a probability simplex.
    B = rng.dirichlet(alpha=np.full(VOCAB, 0.4), size=STATES)
    # initial state distribution.
    pi = rng.dirichlet(alpha=np.full(STATES, 0.5))
    return A, B, pi

rng = np.random.default_rng(SEED)
HMMS = [make_hmm(rng) for _ in range(N_TASKS)]

def sample_seq(hmm, T=T, rng=None):
    A, B, pi = hmm
    rng = rng or np.random.default_rng()
    s = rng.choice(STATES, p=pi)
    out = []
    for _ in range(T):
        v = rng.choice(VOCAB, p=B[s])
        out.append(v)
        s = rng.choice(STATES, p=A[s])
    return np.array(out, dtype=np.int64)

# Pretraining dataset.
N_TRAIN = 20000
train_seqs = np.zeros((N_TRAIN, T), dtype=np.int64)
for i in range(N_TRAIN):
    k = rng.integers(0, N_TASKS)
    train_seqs[i] = sample_seq(HMMS[k], rng=rng)
print('train shape:', train_seqs.shape)

## Exact Bayes-optimal next-symbol predictor under the mixture

For each task $k$, run the HMM forward algorithm to get $p_k(\text{prefix})$ and $p_k(\text{next} \mid \text{prefix})$. The mixture predictor is
$$ p(\text{next} \mid \text{prefix}) = \sum_k p(k \mid \text{prefix}) \, p_k(\text{next} \mid \text{prefix}), \quad p(k \mid \text{prefix}) \propto p_k(\text{prefix}). $$
We implement this in a numerically stable way using log-probabilities.

In [ ]:
def log_forward(hmm, seq):
    """Return list of log p(x_{1..t}) and final state-belief, for t=1..T."""
    A, B, pi = hmm
    logA = np.log(A + 1e-30)
    logB = np.log(B + 1e-30)
    logpi = np.log(pi + 1e-30)
    T_ = len(seq)
    alpha = logpi + logB[:, seq[0]]
    logZs = [None] * T_
    log_total = np.logaddexp.reduce(alpha)
    logZs[0] = log_total
    for t in range(1, T_):
        alpha = np.logaddexp.reduce(alpha[:, None] + logA, axis=0) + logB[:, seq[t]]
        log_total = np.logaddexp.reduce(alpha)
        logZs[t] = log_total
    # next-symbol prob at each t = p(x_{t+1} | x_{1..t}); we'll compute below.
    return alpha, logZs

def next_log_prob(hmm, seq):
    """For each t in 0..T-1, return log p(x_{t+1}=v | x_{1..t}) under this task."""
    A, B, pi = hmm
    logA = np.log(A + 1e-30); logB = np.log(B + 1e-30); logpi = np.log(pi + 1e-30)
    T_ = len(seq)
    alpha = logpi + logB[:, seq[0]]
    out = np.zeros((T_, VOCAB))
    out[0] = np.logaddexp.reduce((np.logaddexp.reduce(alpha[:, None] + logA, axis=0))[:, None] + logB, axis=0)
    out[0] = out[0] - np.logaddexp.reduce(out[0])
    for t in range(1, T_):
        alpha = np.logaddexp.reduce(alpha[:, None] + logA, axis=0) + logB[:, seq[t]]
        s_next = np.logaddexp.reduce(alpha[:, None] + logA, axis=0)            # log p(s_{t+1} | x_{1..t})
        log_p = np.logaddexp.reduce(s_next[:, None] + logB, axis=0)            # log p(x_{t+2}=v | x_{1..t})
        out[t] = log_p - np.logaddexp.reduce(log_p)
    return out  # shape (T, VOCAB), normalized

def bayes_optimal(seq):
    log_priors = []
    next_logs = []
    for hmm in HMMS:
        _, logZs = log_forward(hmm, seq)
        log_priors.append(np.array(logZs))           # log p_k(x_{1..t})
        next_logs.append(next_log_prob(hmm, seq))    # log p_k(x_{t+1}=v | x_{1..t})
    log_priors = np.stack(log_priors, axis=0)         # (K, T)
    next_logs = np.stack(next_logs, axis=0)           # (K, T, V)
    # log p(k) = log(1/K)
    log_pk_prior = -math.log(N_TASKS)
    # log p(k | x_{1..t}) ∝ log p_k(x_{1..t}) + log p(k)
    log_post = log_priors + log_pk_prior
    log_post -= np.logaddexp.reduce(log_post, axis=0)[None, :]   # normalize across K
    # mixture predictor in log-space:
    mix = np.logaddexp.reduce(log_post[:, :, None] + next_logs, axis=0)  # (T, V)
    mix -= np.logaddexp.reduce(mix, axis=1)[:, None]                     # normalize across V
    return mix  # shape (T, V), log-prob

## A tiny transformer trained on next-token prediction

**Prediction.** As context length grows, the trained model's next-token distribution should approach the Bayes-optimal mixture predictor's distribution: KL divergence should shrink with $t$.

In [ ]:
D_MODEL = 64; N_HEADS = 4; N_LAYERS = 2

class TinyTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB, D_MODEL)
        self.pos_emb = nn.Embedding(T, D_MODEL)
        layer = nn.TransformerEncoderLayer(d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=4*D_MODEL,
                                            batch_first=True, dropout=0.0, norm_first=True)
        self.tr = nn.TransformerEncoder(layer, num_layers=N_LAYERS)
        self.head = nn.Linear(D_MODEL, VOCAB)
        mask = torch.triu(torch.full((T, T), float('-inf')), diagonal=1)
        self.register_buffer('causal_mask', mask)

    def forward(self, x):
        b, t = x.shape
        pos = torch.arange(t, device=x.device).unsqueeze(0).expand(b, -1)
        h = self.tok_emb(x) + self.pos_emb(pos)
        h = self.tr(h, mask=self.causal_mask[:t, :t])
        return self.head(h)

model = TinyTransformer().to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)
BATCH = 128; STEPS = 1500
data = torch.from_numpy(train_seqs).to(DEVICE)

for step in range(STEPS):
    idx = torch.randint(0, N_TRAIN, (BATCH,), device=DEVICE)
    x = data[idx]                # (B, T)
    logits = model(x[:, :-1])    # (B, T-1, V)
    target = x[:, 1:]
    loss = F.cross_entropy(logits.reshape(-1, VOCAB), target.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if (step + 1) % 300 == 0:
        print(f'step {step+1:4d}  loss = {loss.item():.4f}')

## Compare to Bayes-optimal as a function of context length

For each test sequence, compute the model's predictive distribution at each position and the Bayes-optimal predictive distribution at each position; report KL($\text{Bayes} \,\|\, \text{model}$) averaged over many test sequences, as a function of position.

In [ ]:
N_TEST = 400
kls = np.zeros((N_TEST, T - 1))
model.eval()
for i in range(N_TEST):
    k = rng.integers(0, N_TASKS)
    seq = sample_seq(HMMS[k], rng=rng)
    with torch.no_grad():
        x = torch.tensor(seq, dtype=torch.long, device=DEVICE).unsqueeze(0)
        logits = model(x[:, :-1]).squeeze(0)
        log_q = F.log_softmax(logits, dim=-1).cpu().numpy()    # (T-1, V)
    log_p = bayes_optimal(seq)[:-1]                            # (T-1, V)
    p = np.exp(log_p)
    kl = np.sum(p * (log_p - log_q), axis=-1)                  # KL(p||q) at each position
    kls[i] = kl

mean_kl = kls.mean(axis=0)
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(np.arange(1, T), mean_kl)
ax.set_xlabel('position in sequence')
ax.set_ylabel(r'$\mathrm{KL}(p_{\mathrm{bayes}} \,||\, p_{\mathrm{model}})$')
ax.set_title('Trained-transformer predictions approach Bayes-optimal with context')
fig.tight_layout()
fig.savefig('figures/04/kl_vs_position.png', dpi=120)
plt.show()
print('first-position KL:', mean_kl[0], '  last-position KL:', mean_kl[-1])

## What this shows. What this does not show. What would refute it.

**What this shows.** On a synthetic mixture-of-4-HMMs, a tiny transformer trained on next-token prediction approaches the exact Bayes-optimal predictor's distribution as context length grows. The mean KL from Bayes to model decreases substantially with position — the empirical signature predicted by Xie et al. (2022). This is *constructive* support for the implicit-Bayes view of ICL in the toy regime.

**What this does not show.** Natural-language ICL is not a mixture of 4 HMMs; the deviations from Bayes-optimality on real data (Panwar et al. 2023) are second-order effects that a 4-HMM toy cannot exhibit. We have also not run long enough to see the *transient* regime (Singh et al. 2023) where ICL emerges and then disappears in favor of in-weights learning — that would require much longer training and a careful protocol.

**What would refute the toy claim.** The model's KL from Bayes-optimal stays *flat* (no improvement with context), or *increases* with context. Either would mean the trained model is not implementing approximately-Bayesian ICL on the mixture.

**Where to go next.** Read Xie et al. (2022) for the original GINC framing, then Panwar et al. (2023) for where the Bayesian frame on natural-data ICL succeeds and fails. The synthesis essay [`circuits-and-icl-bayes`](../essays/circuits-and-icl-bayes.md) connects this notebook's content to the induction-head notebook.